# Scrapping the Delhi School of Economics, Winter School [Website](http://econdse.org/winter-school-2023/)

> Add blockquote


**Note:** For reproducibility:



1.   First make a copy of the notebook and work on the copy
2.   Check these things: paths name, file names, APIs IDs, and customize them according to your environment
3.   After doing a major data operation, you can push your final version here: https://github.com/GaelKBertrand/scholar_info_append






### Mounting Google Drive as a working director

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Installing and loading all dependencies and libraries needed

In [2]:
#Install necessary dependencies and libraries
!pip install waybackpy
!pip install googlesearch-python
!pip install google-api-python-client
!pip install beautifulsoup4
!pip install fake_useragent #necessary for google search rapid automation
!pip install selenium
!apt-get update
!apt install -y chromium-chromedriver
!apt-get install -y wget
!pip install requests-html

import requests
from bs4 import BeautifulSoup
import pandas as pd
from time import sleep
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import logging


from waybackpy import WaybackMachineSaveAPI
from googlesearch import search
from googleapiclient.discovery import build
import time
import requests
from fake_useragent import UserAgent
from bs4 import BeautifulSoup
import logging

#from requests_html import HTMLSession
#from requests_html import AsyncHTMLSession
import pandas as pd
import logging
import asyncio
import nest_asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.7/481.7 kB 29.2 MB/s eta 0:00:00
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:3 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:4 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:6 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,185 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-

# Scrapping the Website

## Using beautifulsoup

In [3]:
# Install necessary libraries
!pip install requests beautifulsoup4 pandas

# Import required libraries
import requests
from bs4 import BeautifulSoup
import pandas as pd


In [4]:
# Function to fetch and scrape the data
def scrape_conference_info(url):
    # Fetch the HTML content from the URL
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')

    # Create a list to store the scraped data
    conference_data = []

    # Extract the conference year (assuming it's in the title or similar place)
    title_tag = soup.find('title')
    if title_tag:
        conference_year = title_tag.text.split()[-1]  # Extract year from title
    else:
        conference_year = "Unknown Year"

    # Extract session names, paper titles, authors, and presenters
    # We will search for specific patterns in the HTML
    sessions = soup.find_all('tr', {'bgcolor': 'white'})  # Find all session rows

    for session in sessions:
        # Find time, location, and event details for each session
        columns = session.find_all('td')

        if len(columns) >= 3:  # If the session has all 3 columns
            time = columns[0].text.strip()
            location = columns[1].text.strip()
            event = columns[2].text.strip()

            # Paper title (likely to be a link or in specific patterns, refine based on actual structure)
            if 'Invited Talk' in event or 'Parallel Session' in event:
                paper_title = event.split(':')[-1].strip()  # Get paper title after the colon
                # Extract authors (if available, this is an example)
                authors = paper_title.split(",")  # In real use, modify how authors are extracted
                presenter = event.split('presented by')[-1].strip() if 'presented by' in event else "Unknown"
                session_name = "Invited Talk" if "Invited" in event else "Parallel Session"

                # Append the data to our list
                conference_data.append([conference_year, session_name, paper_title, ", ".join(authors), presenter])

    # Create a DataFrame from the scraped data
    df = pd.DataFrame(conference_data, columns=['Conference Year', 'Session Name', 'Paper Title', 'Authors', 'Presenter'])

    return df

In [8]:
# Provide the URL of the HTML page to scrape
url = "https://editorialexpress.com/conference/WS2023/program/WS2023.html"  # Replace with the actual URL

# Scrape the data and get it in a DataFrame
df = scrape_conference_info(url)

# Display the results in a DataFrame
df.head()

,Conference Year,Session Name,Paper Title,Authors,Presenter
0,2023,Invited Talk,"John Wooders, NYU Abu Dhabi, Swami Vivekananda...","John Wooders, NYU Abu Dhabi, Swami Vivekanan...",Unknown
1,2023,Parallel Session,Day 1 Parallel Sessions 1,Day 1 Parallel Sessions 1,Unknown
2,2023,Invited Talk,"John Wooders, NYU Abu Dhabi, Swami Vivekananda...","John Wooders, NYU Abu Dhabi, Swami Vivekanan...",Unknown
3,2023,Parallel Session,Day 1 Parallel Sessions 2,Day 1 Parallel Sessions 2,Unknown
4,2023,Invited Talk,"The Neoclassical Perspective, Swami Vivekanand...","The Neoclassical Perspective, Swami Vivekanan...",Unknown


In [10]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Step 1: Load the URL (replace with your desired URL)
url = 'https://editorialexpress.com/conference/WS2023/program/WS2023.html'  # Replace with your specific URL

# Fetch the page content
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')

# Step 2: Parse the required data
conference_year = "2023"  # Extracted directly from the context

sessions = []
papers = []
authors = []
presenters = []

# Find all rows of the schedule (identified by 'tr' elements in the HTML)
for row in soup.find_all('tr', bgcolor='white'):
    time_cell = row.find('td', align='left')  # Get the time column
    event_cells = row.find_all('td', align='left')

    if len(event_cells) > 2:
        event_cell = event_cells[2]  # Event details column
        event_text = event_cell.get_text(strip=True)

        # Ensure that the length of the lists is consistent for each row
        session = ""
        paper = ""
        author = ""
        presenter = ""

        # Extract session names, paper titles, authors, and presenters
        if 'Day' in event_text:
            # Session name extraction (e.g., Day 1 Registration, Inaugural Session, etc.)
            session = event_text
        elif 'Invited Talk' in event_text or 'Parallel Sessions' in event_text:
            # Paper title and presenters (these are links with the detail anchors)
            paper_title = event_text.split('by')[0].strip() if 'by' not in event_text else event_text.split('by')[0].strip()
            presenters_name = event_text.split('by')[-1].strip() if 'by' in event_text else ''

            paper = paper_title
            presenter = presenters_name

        sessions.append(session)
        papers.append(paper)
        authors.append("")  # Author extraction could be more complex based on the HTML structure
        presenters.append(presenter)

# Step 3: Create DataFrame to organize the data
data = {
    'Conference Year': [conference_year] * len(sessions),
    'Session Name': sessions,
    'Paper Title': papers,
    'Author(s)': authors,
    'Presenter': presenters
}

# Ensure all lists are of the same length
max_len = max(len(sessions), len(papers), len(authors), len(presenters))
sessions += [''] * (max_len - len(sessions))
papers += [''] * (max_len - len(papers))
authors += [''] * (max_len - len(authors))
presenters += [''] * (max_len - len(presenters))

df = pd.DataFrame(data)

# Display the results
df


,Conference Year,Session Name,Paper Title,Author(s),Presenter
0,2023,"Day 1 Registration, Foyer",,,
1,2023,,,,
2,2023,"Day 1 Invited Talk 1: John Wooders, NYU Abu Dh...",,,
3,2023,,,,
4,2023,Day 1 Parallel Sessions 1,,,
5,2023,,,,
6,2023,"Day 1 Invited Talk 2: John Wooders, NYU Abu Dh...",,,
7,2023,,,,
8,2023,Day 1 Parallel Sessions 2,,,
9,2023,"Day 2 Registration, Foyer",,,


## Using Requests

In [15]:
!pip install beautifulsoup4
!pip install pandas
!pip install requests

import requests
from bs4 import BeautifulSoup
import pandas as pd


In [29]:
url = "https://editorialexpress.com/conference/WS2023/program/WS2023.html"
response = requests.get(url)
html_content = response.text


In [22]:
soup = BeautifulSoup(html_content, 'html.parser')

# Extract the conference year from the title
conference_year = soup.find('title').text.split()[-1]

# Initialize lists to store the extracted data
session_names = []
paper_titles = []
authors = []
presenters = []

# Extract session names, paper titles, authors, and presenters
for session_header in soup.find_all('tr', bgcolor='lightgrey'):
    current_session_name = session_header.find('th').text.strip()

    next_sibling = session_header.find_next_sibling('tr')
    while next_sibling and next_sibling.get('bgcolor') != 'lightgrey':
        if next_sibling.get('bgcolor') == 'white':
            event_tds = next_sibling.find_all('td')
            if len(event_tds) == 3:  # Ensure it has 3 columns
                event_text = event_tds[-1].get_text(separator=" ").strip()

                # Extract paper title, authors and presenter
                title_match = re.search(r'([^:]+):\s*"([^"]+)"', event_text)
                author_match = re.findall(r'([A-Za-z\s]+)\s?\(([^)]+)\)', event_text)
                presenter_match = re.search(r'presented by ([A-Za-z\s]+)', event_text)

                if title_match:
                    paper_title = title_match.group(2)
                    authors_list = ', '.join([f"{name} ({affil})" for name, affil in author_match])
                    presenter = presenter_match.group(1) if presenter_match else ''

                    session_names.append(current_session_name)
                    paper_titles.append(paper_title)
                    authors.append(authors_list)
                    presenters.append(presenter)
        next_sibling = next_sibling.find_next_sibling('tr')

# Create a DataFrame
df = pd.DataFrame({
    'Conference Year': conference_year,
    'Session Name': session_names,
    'Paper Title': paper_titles,
    'Authors': authors,
    'Presenter': presenters
})

# Display the DataFrame
print(df)


Empty DataFrame
Columns: [Conference Year, Session Name, Paper Title, Authors, Presenter]
Index: []


In [ ]:
# Save the DataFrame to a CSV file
df.to_csv('/content/conference_details.csv', index=False)

In [32]:


# Step 2: Initialize variables and lists
conference_year = "2023"
session_names = []
paper_titles = []
authors = []
presenters = []

lines = html_content.split('\n')
current_session_name = None

# Step 3: Parse the HTML content line by line
for i in range(len(lines)):
    line = lines[i].strip()

    # Extract session name
    if 'bgcolor="lightgrey"' in line:
        current_session_name = line.split('>')[1].split('<')[0].strip()

    # Extract paper details
    elif 'bgcolor="white"' in line:
        if '<td>' in line and 'presented by' in lines[i + 1]:
            next_line = lines[i + 1].strip()
            parts = next_line.split(':')
            paper_title = parts[1].split('"')[1].strip() if len(parts) > 1 else ''
            presenter = next_line.split('presented by')[-1].strip().split('<')[0].strip() if 'presented by' in next_line else ''
            author_list = parts[0].replace('presented by', '').strip()

            session_names.append(current_session_name)
            paper_titles.append(paper_title)
            authors.append(author_list)
            presenters.append(presenter)

# Step 4: Create DataFrame and Save to CSV
df = pd.DataFrame({
    'Conference Year': [conference_year] * len(session_names),
    'Session Name': session_names,
    'Paper Title': paper_titles,
    'Authors': authors,
    'Presenter': presenters
})

# Display the DataFrame
print(df)

# Save the DataFrame to a CSV file
df.to_csv('/content/conference_details.csv', index=False)


Empty DataFrame
Columns: [Conference Year, Session Name, Paper Title, Authors, Presenter]
Index: []


To delete

In [ ]:
# Load HTML from a URL (replace 'your_url' with the actual URL)
url = 'https://www.isleijle.org/isle-conference-archives/'
response = requests.get(url)
html_content = response.content


In [ ]:
# Parse the HTML content
soup = BeautifulSoup(html_content, 'html.parser')

# Initialize lists for extracted data
conference_years = np.array([])
volumes = []
issues = []
titles = []
authors = []

# Locate all <h2> tags with class "et_pb_toggle_title" (conference year)
conference_year_tags = soup.find_all('h2', class_='et_pb_toggle_title')


In [ ]:
import numpy as np

# Extract data for each conference year
for year_tag in conference_year_tags:
    year = year_tag.text.strip()  # Extract the year
    toggle_content = year_tag.find_next_sibling('div', class_='et_pb_toggle_content clearfix')  # Get sibling div

    if toggle_content:
        # Locate all rows inside the content section
        rows = toggle_content.find_all('div', class_='et_pb_text_inner')
        for row in rows:
            # Extract volume and issue
            volume_issue_header = row.find('h2')
            if volume_issue_header:
                volume_issue = volume_issue_header.text.strip()
                volume, issue = volume_issue.split('::')[1].split('Issue') if 'Issue' in volume_issue else (volume_issue, None)
                volume = volume.strip() if volume else None
                issue = issue.strip() if issue else None

            # Extract individual papers
            papers = row.find_all('p')
            for paper in papers:
                title_tag = paper.find('strong')  # Find the paper title
                author_tag = paper.find('em')  # Find the author

                title = title_tag.text.strip() if title_tag else None
                author = author_tag.text.strip() if author_tag else None

                # Append the data to lists
                # Convert 'Conference Year' to string to handle it properly
                # Convert 'conference_years' to a NumPy array if it's a list
                conference_years = np.array(conference_years)
                # Now you can use .astype()
                conference_years = conference_years.astype(str)
                conference_years = np.append(conference_years, year)
                volumes.append(volume)
                issues.append(issue)
                titles.append(title)
                authors.append(author)

# Create a DataFrame from the extracted data
data = pd.DataFrame({
    'Conference Year': conference_years,
    'Volume': volumes,
    'Issue Number': issues,
    'Paper Title': titles,
    'Author': authors
})


In [ ]:
# Display the DataFrame
print(data)

# Save to CSV if needed
data.to_csv('conference_data.csv', index=False)

     Conference Year     Volume Issue Number  \
0               1958   Volume 1          1-2   
1               1958   Volume 1          1-2   
2               1958   Volume 1          1-2   
3               1958   Volume 1          1-2   
4               1958   Volume 1          1-2   
...              ...        ...          ...   
2020            2007  Volume 50            4   
2021            2007  Volume 50            4   
2022            2007  Volume 50            4   
2023            2007  Volume 50            4   
2024            2007  Volume 50            4   

                                            Paper Title  \
0     Wage and Employment Policy of Trade Union in a...   
1                                                  None   
2           Disguised Unemployment or Underemployment ?   
3           Wage Policy and Economic Planning in India,   
4            The Problem of Disparity in Wage , Incomes   
...                                                 ...   
2020  Wage

In [ ]:
from bs4 import BeautifulSoup
import pandas as pd
import requests

# Load HTML from a URL (replace 'your_url' with the actual URL)
url = 'https://www.isleijle.org/isle-conference-archives/'
response = requests.get(url)
html_content = response.content

# Parse the HTML content
soup = BeautifulSoup(html_content, 'html.parser')

# Initialize lists for extracted data
conference_years = []
volumes = []
issues = []
titles = []
authors = []

# Locate all <h2> tags with class "et_pb_toggle_title" (conference year)
conference_year_tags = soup.find_all('h2', class_='et_pb_toggle_title')

# Extract data for each conference year
for year_tag in conference_year_tags:
    year = year_tag.text.strip()  # Extract the year
    toggle_content = year_tag.find_next_sibling('div', class_='et_pb_toggle_content clearfix')  # Get sibling div

    if toggle_content:
        # Locate all rows inside the content section
        rows = toggle_content.find_all('div', class_='et_pb_text_inner')
        for row in rows:
            # Extract volume and issue
            volume_issue_header = row.find('h2')
            if volume_issue_header:
                volume_issue = volume_issue_header.text.strip()
                volume, issue = volume_issue.split('::')[1].split('Issue') if 'Issue' in volume_issue else (volume_issue, None)
                volume = volume.strip() if volume else None
                issue = issue.strip() if issue else None

            # Extract individual papers
            papers = row.find_all('p')
            for paper in papers:
                title_tag = paper.find('strong')  # Find the paper title
                author_tag = paper.find('em')  # Find the author

                title = title_tag.text.strip() if title_tag else None
                author = author_tag.text.strip() if author_tag else None

                # Clean the title to remove any newline or extra spaces (multi-line titles fix)
                if title:
                    title = " ".join(title.splitlines()).strip()

                # Append the data to lists
                conference_years.append(year)
                volumes.append(volume)
                issues.append(issue)
                titles.append(title)
                authors.append(author)

# Create a DataFrame from the extracted data
data = pd.DataFrame({
    'Conference Year': conference_years,
    'Volume': volumes,
    'Issue Number': issues,
    'Paper Title': titles,
    'Author': authors
})

# Display the DataFrame
print(data)

# Save to CSV if needed
data.to_csv('conference_data2.csv', index=False)


     Conference Year     Volume Issue Number  \
0               1958   Volume 1          1-2   
1               1958   Volume 1          1-2   
2               1958   Volume 1          1-2   
3               1958   Volume 1          1-2   
4               1958   Volume 1          1-2   
...              ...        ...          ...   
2020            2007  Volume 50            4   
2021            2007  Volume 50            4   
2022            2007  Volume 50            4   
2023            2007  Volume 50            4   
2024            2007  Volume 50            4   

                                            Paper Title  \
0     Wage and Employment Policy of Trade Union in a...   
1                                                  None   
2           Disguised Unemployment or Underemployment ?   
3           Wage Policy and Economic Planning in India,   
4            The Problem of Disparity in Wage , Incomes   
...                                                 ...   
2020  Wage

Check out missing values

In [ ]:
import pandas as pd

# Load the existing CSV file (replace 'conference_data.csv' with your actual file path)
existing_data = pd.read_csv('conference_data.csv')

# Filter the DataFrame to include only rows where the 'Conference Year' is 2004
data_2004 = existing_data[existing_data['Conference Year'] == 2004]

# Print the filtered data
print(data_2004)


      Conference Year     Volume Issue Number Paper Title Author
1775             2004  Volume 47            1         NaN    NaN
1776             2004  Volume 47            1         NaN    NaN
1777             2004  Volume 47            1         NaN    NaN
1778             2004  Volume 47            1         NaN    NaN
1779             2004  Volume 47            1         NaN    NaN
...               ...        ...          ...         ...    ...
1832             2004  Volume 47            4         NaN    NaN
1833             2004  Volume 47            4         NaN    NaN
1834             2004  Volume 47            4         NaN    NaN
1835             2004  Volume 47            4         NaN    NaN
1836             2004  Volume 47            4         NaN    NaN

[62 rows x 5 columns]


Save to drive

In [ ]:
# Save the updated DataFrame back to CSV
existing_data.to_csv('updated_conference_data.csv', index=False)

Manual Work to be done to correct for these cases:



1.   Empty cells in the Author Column
2.   Empty cells in the Paper Title Column



Count the authors

Note: Load the files after dealing with missing values manually

In [ ]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv('/content/drive/My Drive/ISLE.csv')

# Function to clean and count authors
def clean_and_count_authors(row):
    # If 'Author' is NaN or empty, return 0 authors
    if pd.isna(row['Author']) or row['Author'].strip() == "":
        return 0

    # Remove any leading commas
    author_names = re.sub(r'^\s*,\s*', '', row['Author'])

    # Split by comma to count the number of authors
    authors = author_names.split(',')

    # Update 'Number of Authors' based on the comma-separated names
    return len(authors)

# Apply the cleaning function to the 'Author' column and update 'Number of Authors'
df['Number of Authors'] = df.apply(clean_and_count_authors, axis=1)

# Drop the 'Co-Authors' column
df.drop(columns=['Co-Authors'], inplace=True)

# Save the resulting dataset
df.to_csv('/content/drive/My Drive/ISLE_clean.csv', index=False)

# If you want to view the updated dataset
# print(df.head())


In [ ]:
print(df.head())

   Conference Year    Volume Issue Number  \
0             1958  Volume 1          1-2   
1             1958  Volume 1          1-2   
2             1958  Volume 1          1-2   
3             1958  Volume 1          1-2   
4             1958  Volume 1          1-2   

                                         Paper Title            Author  \
0  Wage and Employment Policy of Trade Union in a...      Banerjee P C   
1            Housing the Industrial Worker in India,        Saxena S P   
2        Disguised Unemployment or Underemployment ?         Singh V B   
3        Wage Policy and Economic Planning in India,  Srivastava H G P   
4         The Problem of Disparity in Wage , Incomes     Upadhyaya P K   

   Number of Authors  
0                  1  
1                  1  
2                  1  
3                  1  
4                  1  
